# 15-minute cities — Nigeria

Adaptation of Bruno, Melo, Campanelli & Loreto’s metrics (PT, F15, Gini, N\*) to five Nigerian urban centres. *Nature Cities* (2024). doi:[10.1038/s44284-024-00119-4](https://doi.org/10.1038/s44284-024-00119-4). **This is not a replication.** Bruno’s protocol (GHS urban centres, nine OSM categories, dual access at n = 20, OSRM) is not a feasible headline here.

This notebook is the working document for **five urban centres**: Lagos, Kano, Ibadan, Abuja, and Port Harcourt.

## Headline vs stress test (do not mix them in a sentence)

| | Question | Data |
|---|---|---|
| **Headline** | What mapped 15-minute access looks like with local POIs and a local walk graph | Metro LGAs, GRID3 health and schools, dual access *n* = 5, OSM walk at 5 km/h, GRID3/WorldPop NGA v3.0 |
| **Stress test** | What happens if we copy Bruno’s *n* | Same hexes and POIs, dual access *n* = 20 (`robustness.csv`). Not a clone of their paper. |

The Sony CSL [atlas](https://whatif.sonycsl.it/15mincity/) (Lagos `idcity=4800`) is a **baseline**, not the finding. Observed F15 is **mapped** 15-minuteness. When POIs are sparse, lead with density and N\*.

## What the pipeline computes now

GRID3 health and education on a 200 m hex grid, population from GRID3/WorldPop NGA v3.0, dual-access proximity time at *n* = 5, city-level **PT**, **F15**, **Gini(PT)**, completeness mask, N\*, and framed maps. Headline walk times are the **OSM walk graph at 5 km/h**. Euclidean stays as a diagnostic column — never quote `F15_eucl` as a result.

## Metric trace (raw → number)

1. **Boundary.** Metro LGAs in `cities.py` (Ibadan is the core 5, not 11). Not GHS Urban Centre.
2. **Grid.** Pointy-top hexagons, side 200 m, clipped to that boundary.
3. **Population.** Sum of GRID3 / WorldPop NGA v3.0 cells in each hex (`pop_k`).
4. **POIs.** GRID3 schools (national) and GRID3 health: v3 where it exists (Kano, Ibadan, Abuja), v2 for Lagos and Port Harcourt. OSM is a completeness overlay, not the amenity census.
5. **Dual access.** For category *c*, mean time to the *n* nearest facilities from the hex centroid. Headline uses *n* = 5 for health and education (they are not interchangeable the way cafés are).
6. **PT_k.** Unweighted mean of the category times that exist in that hex.
7. **PT_city.** Population-weighted mean of PT_k over mapped hexes.
8. **F15.** Share of *mapped* population with PT_k ≤ 15. The unmapped share is reported beside it, never dropped silently.
9. **Gini(PT).** Inequality of person-weighted PT_k.

Bruno’s nine OSM amenity groups are a stress test, not the headline basket. N\* and the OSM walk graph live in `src/proximity/`, not this notebook.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
for extra in (ROOT / ".pydeps", ROOT / "src"):
    sys.path.insert(0, str(extra))

from proximity.cities import CITIES, HEX_SIDE_M, N_DUAL_NONSUBSTITUTABLE, WALK_M_PER_MIN
from proximity.download import DATASETS
from proximity.pipeline import run

print("Cities:", ", ".join(c.name for c in CITIES.values()))
print(f"Hex side {HEX_SIDE_M} m · dual-access n={N_DUAL_NONSUBSTITUTABLE} · walk {WALK_M_PER_MIN:.1f} m/min")

## Data provenance (HDX first)

GRID3 settlement extents were **not** downloaded (2 GB). We do not need the national block layer to hex a city.

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {"layer": k, "source": v["source"], "vintage": v["vintage"], "note": v["note"]}
        for k, v in DATASETS.items()
    ]
)

## Run the five-city pipeline

Downloads skip files that already exist under `data/raw/` (gitignored). Maps write to `maps/` as PNG and PDF. Hex layers write to `data/processed/{city}_hexes.gpkg`.

In [ ]:
table = run()
table[[
    "city", "PT_city", "F15", "Gini_PT",
    "pop_total", "pop_unmapped_share",
    "health_n", "school_n", "health_source", "n_hex",
]]

### How to read the table

- **PT_city** is minutes. Lower is closer. Euclidean times are optimistic relative to street networks with lagoons, expressways, and walls.
- **F15** is only the mapped population. If `pop_unmapped_share` is large, the city is under-inventoried, not empty of services.
- **health_source** tells you whether GRID3 actually covers that city. Treat HOT OSM fallback as a completeness warning.

Open `maps/{city}_PT_k.png` for the framed choropleth (title, north arrow, scale bar, classed teal legend, hatch for unmapped hexes).

In [ ]:
from IPython.display import Image, display
from proximity.paths import MAPS

for slug in CITIES:
    for suffix in ("centre_minutes", "PT_k"):
        png = MAPS / f"{slug}_{suffix}.png"
        if png.exists():
            display(Image(filename=str(png), width=640))

## Next (not in this notebook)

1. Optional **GHS Urban Centre** clip as a sensitivity on the same metrics — not a Bruno clone, and not the headline boundary.
2. Food / water / worship in the headline basket, or retitle the paper as clinic-and-school access.
3. N\* maps (the table is already in `nstar.csv`).
4. Refresh this MapLibre story with walk `PT_k` hexes, not only density.